<a href="https://www.kaggle.com/code/gamzegezgin/suspicious-activities-graduation-project?scriptVersionId=311089567" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# **Importing the necessary libraries**

In [ ]:
pip install tensorflow-io -q

In [ ]:
import os
import cv2
import re
import math
import random
import numpy as np
import datetime as dt
import tensorflow as tf
from collections import deque
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import *
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing.image import ImageDataGenerator

## Veri Seti ve Parametreler

In [ ]:
# Sabitler
seed_constant = 3
np.random.seed(seed_constant)
random.seed(seed_constant)
tf.random.set_seed(seed_constant)

# Sınıflar
classes = ["Fighting", "Vandalism", "NormalVideos"]

# Dataset path
dataset_path = "/kaggle/input/datasets/odins0n/ucf-crime-dataset/Train"

# Model parametreleri
img_height = 64
img_width  = 64
seq_len    = 30

print("Sınıflar:", classes)
print("Dataset path:", dataset_path)

In [ ]:
import os

dataset_path = "/kaggle/input/datasets/odins0n/ucf-crime-dataset/Train"

print("=" * 50)
for cls in classes:
    path = os.path.join(dataset_path, cls)
    files = sorted(os.listdir(path))
    
    # Unique video sayısını bul
    import re
    videos = set()
    for f in files:
        match = re.match(r'(.+)_(\d+)\.png$', f)
        if match:
            videos.add(match.group(1))
    
    total_videos = len(videos)
    used = 30
    pct = (used / total_videos) * 100
    print(f"{cls}:")
    print(f"  Toplam video: {total_videos}")
    print(f"  Kullandığımız: {used} ({pct:.1f}%)")
print("=" * 50)

## Sınıf Dağılımı

In [ ]:
# Her sınıftaki frame sayısı
for cls in classes:
    path = os.path.join(dataset_path, cls)
    files = sorted(os.listdir(path))
    print(f"{cls}: {len(files)} frame")

In [ ]:
## Dataset Oluşturma (PNG Frame'lerden Sequence)

In [ ]:
def get_video_groups(class_path, max_videos=30):
    """Aynı videoya ait frame'leri grupla"""
    files = sorted(os.listdir(class_path))
    groups = {}
    for f in files:
        match = re.match(r'(.+)_(\d+)\.png$', f)
        if match:
            video_name = match.group(1)
            frame_no   = int(match.group(2))
            if video_name not in groups:
                groups[video_name] = []
            groups[video_name].append((frame_no, f))
    for v in groups:
        groups[v].sort(key=lambda x: x[0])
    return dict(list(groups.items())[:max_videos])


def dataset_creation():
    features = []
    paths    = []
    labels   = []

    datagen = ImageDataGenerator(
        rotation_range=20,
        zoom_range=0.2,
        horizontal_flip=True,
        brightness_range=(0.8, 1.2),
        shear_range=10,
        channel_shift_range=20,
        fill_mode='reflect'
    )

    for index_of_class, name_of_class in enumerate(classes):
        print(f'Extracting Class: {name_of_class}')
        class_path   = os.path.join(dataset_path, name_of_class)
        video_groups = get_video_groups(class_path, max_videos=30)
        print(f'  {len(video_groups)} video grubu')

        for video_name, frame_list in video_groups.items():
            total = len(frame_list)
            if total < seq_len:
                continue
            step = max(total // seq_len, 1)
            selected = frame_list[::step][:seq_len]
            if len(selected) < seq_len:
                continue

            sequence = []
            for _, fname in selected:
                img_path = os.path.join(class_path, fname)
                img = cv2.imread(img_path)
                if img is None:
                    break
                img = cv2.resize(img, (img_width, img_height))
                img = datagen.random_transform(img)
                img = img / 255.0
                sequence.append(img)

            if len(sequence) == seq_len:
                features.append(sequence)
                labels.append(index_of_class)
                paths.append(video_name)

    return np.asarray(features), paths, np.array(labels)


features, paths, labels = dataset_creation()
print(f"\nFeatures: {features.shape}, Labels: {len(labels)}")

## Train / Test Ayrımı

In [ ]:
one_hot_encoded_labels = to_categorical(labels)

features_train, features_test, labels_train, labels_test = train_test_split(
    features, one_hot_encoded_labels,
    test_size=0.25, shuffle=True, random_state=seed_constant
)

print(f"Train: {features_train.shape}")
print(f"Test:  {features_test.shape}")

## Model Oluşturma -LRCN+vgg16

In [ ]:
def creating_model():
    model = Sequential()

    vgg = VGG16(include_top=False, weights='imagenet', input_shape=(img_height, img_width, 3))
    for layer in vgg.layers:
        layer.trainable = False

    model.add(TimeDistributed(vgg, input_shape=(seq_len, img_height, img_width, 3)))
    model.add(TimeDistributed(Conv2D(32, (3, 3), padding='same', activation='relu')))
    model.add(TimeDistributed(Conv2D(64, (3, 3), padding='same', activation='relu')))
    model.add(TimeDistributed(Flatten()))
    model.add(LSTM(32))
    model.add(Dense(len(classes), activation='softmax'))

    model.summary()
    return model

## Model Eğitimi (6 Model — Majority Voting Ensemble)

In [ ]:
num_classes   = len(classes)
lstm_models   = []
num_models    = 6
training_histories = []

for i in range(num_models):
    print(f"\n--- Model {i+1}/{num_models} eğitiliyor ---")
    lstm_model = creating_model()
    lstm_model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    early_stopping = EarlyStopping(monitor='accuracy', patience=10, mode='max', restore_best_weights=True)
    history = lstm_model.fit(
        x=features_train, y=labels_train,
        epochs=30, batch_size=4, shuffle=True,
        validation_split=0.25,
        callbacks=[early_stopping]
    )
    lstm_models.append(lstm_model)
    training_histories.append(history)

# Majority voting
y_pred = []
for i in range(len(features_test)):
    preds = [np.argmax(m.predict(np.expand_dims(features_test[i], axis=0), verbose=0)[0]) for m in lstm_models]
    y_pred.append(max(set(preds), key=preds.count))

y_pred = tf.keras.utils.to_categorical(y_pred, num_classes)
print("\nTahminler tamamlandı.")

## Karşılaştırma Metrikleri

In [ ]:
import time
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    f1_score, precision_score, recall_score, accuracy_score
)

y_true = np.argmax(labels_test, axis=1)
y_pred_cls = np.argmax(y_pred, axis=1)

# Temel metrikler
acc  = accuracy_score(y_true, y_pred_cls)
f1   = f1_score(y_true, y_pred_cls, average='weighted')
prec = precision_score(y_true, y_pred_cls, average='weighted')
rec  = recall_score(y_true, y_pred_cls, average='weighted')

print("=" * 50)
print("       DENEY 1 — BAZ MODEL / UCF-CRIME")
print("=" * 50)
print(f"  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)")
print(f"  F1 Score  : {f1:.4f}")
print(f"  Precision : {prec:.4f}")
print(f"  Recall    : {rec:.4f}")

# AUC-ROC
try:
    auc = roc_auc_score(labels_test, y_pred, multi_class='ovr', average='weighted')
    print(f"  AUC-ROC   : {auc:.4f}")
except Exception as e:
    print(f"  AUC-ROC   : hesaplanamadı ({e})")

# False Positive Rate
cm = confusion_matrix(y_true, y_pred_cls)
print("\n--- False Positive Rate (sınıf bazlı) ---")
for i, cls in enumerate(classes):
    fp = cm[:, i].sum() - cm[i, i]
    tn = cm.sum() - cm[i, :].sum() - cm[:, i].sum() + cm[i, i]
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    print(f"  {cls}: {fpr:.4f}")

# FPS
print("\n--- FPS Testi ---")
start = time.time()
for s in features_test[:10]:
    for m in lstm_models:
        m.predict(np.expand_dims(s, axis=0), verbose=0)
fps = 10 / (time.time() - start)
print(f"  FPS: {fps:.2f}")

# Sınıf bazlı rapor
print("\n--- Sınıf Bazlı Rapor ---")
print(classification_report(y_true, y_pred_cls, target_names=classes))
print("=" * 50)

## Görseller

In [ ]:
# Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.title('Confusion Matrix — Baz Model (UCF-Crime)')
plt.ylabel('Gerçek')
plt.xlabel('Tahmin')
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix_deney1.png', dpi=150)
plt.show()

# ROC Eğrisi
plt.figure(figsize=(8, 6))
for i, cls in enumerate(classes):
    try:
        fpr_v, tpr_v, _ = roc_curve(labels_test[:, i], y_pred[:, i])
        plt.plot(fpr_v, tpr_v, label=cls)
    except:
        pass
plt.plot([0,1],[0,1],'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Eğrisi — Baz Model (UCF-Crime)')
plt.legend()
plt.tight_layout()
plt.savefig('/kaggle/working/roc_curve_deney1.png', dpi=150)
plt.show()

print("Görseller kaydedildi: /kaggle/working/")

## Modeli Kaydet

In [ ]:
lstm_models[0].save('/kaggle/working/base_model_deney1.h5')
print("Model kaydedildi.")